# Principal Component Analysis & Manifold Learning
## Notebook 1: Concepts with Synthetic Data

**Topics Covered:**
- What is Dimensionality Reduction?
- PCA Introduction & Intuition
- Manual PCA (from scratch) — Step by Step
- PCA with Scikit-Learn
- Explained Variance, Scree Plot
- Reconstruction (Inverse Transform)
- Manifold Learning — t-SNE
- PCA vs t-SNE Comparison
- Capstone Project

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_blobs, make_classification
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries loaded!")

## 1. What is Dimensionality Reduction?

**Problem:** High-dimensional data is hard to visualize and computationally expensive.

**Solution:** Reduce features while keeping the most important information.

| Aspect | Detail |
|--------|--------|
| **Goal** | Reduce features (columns) while preserving variance/structure |
| **Use Cases** | Visualization, noise removal, feature compression, speed-up |
| **Types** | Linear (PCA) / Non-linear (t-SNE, UMAP) |

**When to use PCA:**
- Too many features (curse of dimensionality)
- Features are correlated (multicollinearity)
- Need to visualize high-dimensional data in 2D/3D
- Want to remove noise from data

## 2. PCA Intuition

**Core Idea:** Find new axes (Principal Components) that capture the **maximum variance** in data.

**Example:** 2D data with correlated X and Y → PCA finds the direction of maximum spread (PC1) and perpendicular direction (PC2).

**Key Concepts:**
- **Principal Component (PC):** A new axis = linear combination of original features
- **PC1:** Direction of maximum variance
- **PC2:** Direction of maximum remaining variance, perpendicular to PC1
- **Loadings:** How much each original feature contributes to each PC
- **Scores:** Data projected onto the new PCA axes

**Math Foundation:**
- **Variance** = how spread out data is
- **Covariance** = how two features change together
- **Eigenvectors** = directions (principal components)
- **Eigenvalues** = amount of variance explained by each PC

```
PC1 = w11*feature1 + w12*feature2 + ... + w1n*featureN
PC2 = w21*feature1 + w22*feature2 + ... + w2n*featureN
```

## 3. Create Synthetic Data

In [ ]:
# Create correlated 2D data to show PCA clearly
np.random.seed(42)
n = 200

# Correlated features (PCA will find the main direction)
mean = [0, 0]
cov = [[3, 2.5], [2.5, 3]]   # High covariance = correlated
X_2d = np.random.multivariate_normal(mean, cov, n)

print(f"Data shape: {X_2d.shape}")
print(f"Feature 1 range: {X_2d[:,0].min():.2f} to {X_2d[:,0].max():.2f}")
print(f"Feature 2 range: {X_2d[:,1].min():.2f} to {X_2d[:,1].max():.2f}")
print(f"Correlation: {np.corrcoef(X_2d[:,0], X_2d[:,1])[0,1]:.3f}")

plt.figure(figsize=(6, 5))
plt.scatter(X_2d[:,0], X_2d[:,1], alpha=0.5, color='steelblue')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Correlated 2D Data (before PCA)')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Manual PCA — Step by Step

**5 Steps of PCA:**
1. **Standardize** the data (zero mean, unit variance)
2. **Compute Covariance Matrix**
3. **Eigendecomposition** (find eigenvectors & eigenvalues)
4. **Sort** eigenvectors by eigenvalues (descending)
5. **Project** data onto top-k eigenvectors

In [ ]:
# ============================================================
# MANUAL PCA — Step by Step
# ============================================================

# STEP 1: Standardize (zero mean)
X_mean = X_2d.mean(axis=0)
X_std = X_2d - X_mean           # subtract mean (centering)
print("Step 1 — Centered data mean:", X_std.mean(axis=0).round(10))

# STEP 2: Covariance Matrix
# Cov(i,j) = how much features i and j vary together
cov_matrix = np.cov(X_std.T)    # shape: (n_features, n_features)
print("\nStep 2 — Covariance Matrix:")
print(cov_matrix.round(3))

# STEP 3: Eigendecomposition
# eigenvectors = principal component directions
# eigenvalues = variance explained by each PC
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
print("\nStep 3 — Eigenvalues:", eigenvalues.round(3))
print("Eigenvectors:\n", eigenvectors.round(3))

# STEP 4: Sort by eigenvalues (largest first)
sort_idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[sort_idx]
eigenvectors = eigenvectors[:, sort_idx]
print("\nStep 4 — Sorted eigenvalues:", eigenvalues.round(3))

# Explained variance ratio
explained_var = eigenvalues / eigenvalues.sum()
print("Explained variance ratio:", explained_var.round(3))
print(f"PC1 explains {explained_var[0]*100:.1f}% of variance")
print(f"PC2 explains {explained_var[1]*100:.1f}% of variance")

# STEP 5: Project data (choose top k=1 PC)
k = 1
W = eigenvectors[:, :k]         # weight matrix (2 x k)
X_pca_manual = X_std @ W        # (n_samples x k)
print(f"\nStep 5 — Projected shape (k={k}): {X_pca_manual.shape}")

In [ ]:
# Visualize original data with principal component directions
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: original data with PC arrows
ax = axes[0]
ax.scatter(X_std[:,0], X_std[:,1], alpha=0.4, color='steelblue')
scale = 2.5
for i, (ev, lbl) in enumerate(zip(eigenvalues, ['PC1','PC2'])):
    vec = eigenvectors[:, i] * np.sqrt(ev) * scale
    ax.annotate('', xy=vec, xytext=[0,0],
                arrowprops=dict(arrowstyle='->', color=['red','orange'][i], lw=2.5))
    ax.text(vec[0]*1.1, vec[1]*1.1, lbl, fontsize=12, color=['red','orange'][i], fontweight='bold')
ax.set_title('Original Data + PCA Directions', fontsize=12)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.axis('equal')
ax.grid(True, alpha=0.3)

# Right: projected data (1D)
ax = axes[1]
ax.scatter(X_pca_manual, np.zeros_like(X_pca_manual), alpha=0.4, color='red')
ax.axhline(0, color='k', lw=0.5)
ax.set_title('Data Projected onto PC1 (1D)', fontsize=12)
ax.set_xlabel('PC1 Score')
ax.set_ylabel('')
ax.set_yticks([])
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Red arrows = PCA directions; PC1 captures most variance!")

## 5. PCA with Scikit-Learn

In [ ]:
# Sklearn PCA is much simpler!
# StandardScaler handles standardization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_2d)

# Fit PCA — keep all components first
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("=== PCA Results ===")
print(f"Original shape:  {X_2d.shape}")
print(f"PCA shape:       {X_pca.shape}")
print(f"\nExplained variance ratio: {pca.explained_variance_ratio_.round(4)}")
print(f"Explained variance (units): {pca.explained_variance_.round(4)}")
print(f"\nPC1 explains: {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"PC2 explains: {pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"Total (cumsum): {pca.explained_variance_ratio_.cumsum().round(4)}")

print(f"\nPrincipal Components (loadings) shape: {pca.components_.shape}")
print("PC1 loadings:", pca.components_[0].round(4))
print("PC2 loadings:", pca.components_[1].round(4))

## 6. PCA for Visualization — Multi-Class Data

In [ ]:
# Generate 4D multi-class data
X_4d, y = make_blobs(n_samples=300, n_features=4, centers=3, random_state=42)
print(f"Original: {X_4d.shape} -> classes: {np.unique(y)}")

# Apply PCA: 4D -> 2D
scaler4 = StandardScaler()
X_4d_scaled = scaler4.fit_transform(X_4d)

pca_2d = PCA(n_components=2)
X_2d_proj = pca_2d.fit_transform(X_4d_scaled)
print(f"After PCA: {X_2d_proj.shape}")
print(f"Variance explained: {pca_2d.explained_variance_ratio_}")
print(f"Total variance: {pca_2d.explained_variance_ratio_.sum()*100:.1f}%")

colors = ['royalblue', 'tomato', 'forestgreen']
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: Original features (feature 1 vs feature 2 only)
for i, c in enumerate(np.unique(y)):
    mask = y == c
    axes[0].scatter(X_4d[mask,0], X_4d[mask,1], color=colors[i], alpha=0.6, label=f'Class {c}')
axes[0].set_title('Original: Feature1 vs Feature2 (2 of 4)', fontsize=11)
axes[0].set_xlabel('Feature 1'); axes[0].set_ylabel('Feature 2')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: PCA projection (captures most variance from ALL 4 features)
for i, c in enumerate(np.unique(y)):
    mask = y == c
    axes[1].scatter(X_2d_proj[mask,0], X_2d_proj[mask,1], color=colors[i], alpha=0.6, label=f'Class {c}')
axes[1].set_title(f'PCA: 4D → 2D ({pca_2d.explained_variance_ratio_.sum()*100:.0f}% variance)', fontsize=11)
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Scree Plot — Choosing n_components

**How many PCs to keep?**
- **Scree plot:** Plot explained variance vs. n_components, look for "elbow"
- **Cumulative explained variance:** Keep PCs until 90-95% variance explained
- **Rule of thumb:** Eigenvalues > 1 (Kaiser criterion)

In [ ]:
# High-dimensional synthetic data
X_high, y_high = make_classification(n_samples=500, n_features=20,
                                      n_informative=8, random_state=42)
scaler_high = StandardScaler()
X_high_scaled = scaler_high.fit_transform(X_high)

# PCA with all components
pca_full = PCA(n_components=20)
pca_full.fit(X_high_scaled)

evr = pca_full.explained_variance_ratio_
cumulative = np.cumsum(evr)

# Find where 90% and 95% variance is explained
n_90 = np.argmax(cumulative >= 0.90) + 1
n_95 = np.argmax(cumulative >= 0.95) + 1
print(f"Components for 90% variance: {n_90}")
print(f"Components for 95% variance: {n_95}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scree plot (individual)
axes[0].bar(range(1, 21), evr, color='steelblue', edgecolor='white')
axes[0].axhline(y=1/20, color='red', linestyle='--', label='Random chance (1/n)')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot (Individual Variance)', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative variance
axes[1].plot(range(1, 21), cumulative, 'o-', color='steelblue', lw=2)
axes[1].axhline(0.90, color='orange', linestyle='--', label='90%')
axes[1].axhline(0.95, color='red', linestyle='--', label='95%')
axes[1].axvline(n_90, color='orange', linestyle=':', alpha=0.7)
axes[1].axvline(n_95, color='red', linestyle=':', alpha=0.7)
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Variance Explained', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Keep {n_90} components → 90% variance retained")

## 8. Reconstruction — Inverse Transform

PCA is **reversible** (approximately). After compressing, you can reconstruct.

**Use case:** Image compression, denoising

In [ ]:
# PCA reconstruction on 20D data
# Compress to k components, then reconstruct
X_scaled_test = X_high_scaled[:5]  # just 5 samples

print("Reconstruction Error at Different n_components:")
print(f"{'n_components':<15} {'MSE':<12} {'% Retained':<15}")
print("-" * 40)
for k in [1, 2, 5, 8, 10, 15, 20]:
    pca_k = PCA(n_components=k)
    X_compressed = pca_k.fit_transform(X_high_scaled)
    X_reconstructed = pca_k.inverse_transform(X_compressed)
    mse = np.mean((X_high_scaled - X_reconstructed)**2)
    var_retained = pca_k.explained_variance_ratio_.sum() * 100
    print(f"{k:<15} {mse:<12.4f} {var_retained:<15.1f}%")

# Visualize reconstruction for one sample
fig, ax = plt.subplots(figsize=(10, 4))
x_original = X_high_scaled[0]
x_positions = range(20)

for k, color in [(2, 'red'), (5, 'orange'), (8, 'green'), (20, 'blue')]:
    pca_k = PCA(n_components=k)
    pca_k.fit(X_high_scaled)
    x_rec = pca_k.inverse_transform(pca_k.transform([x_original]))[0]
    ax.plot(x_positions, x_rec, 'o-', color=color, alpha=0.7, label=f'k={k}', lw=1.5)

ax.plot(x_positions, x_original, 'ks-', lw=2, ms=6, label='Original', zorder=5)
ax.set_xlabel('Feature Index')
ax.set_ylabel('Value')
ax.set_title('Reconstruction Quality vs n_components (Sample 0)', fontsize=12)
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Manifold Learning — Introduction

**PCA limitation:** PCA is **linear** — it finds straight-line projections. Some datasets have **non-linear structure** (like a spiral or swiss roll) that PCA can't capture.

**Manifold Learning:** Non-linear dimensionality reduction that preserves local structure.

| Method | Type | Preserves | Speed | Use Case |
|--------|------|-----------|-------|----------|
| **PCA** | Linear | Global variance | Fast | Correlated features, noise removal |
| **t-SNE** | Non-linear | Local neighborhoods | Slow | Visualization only |
| **UMAP** | Non-linear | Local + global | Medium | Large datasets, visualization |

**t-SNE (t-Distributed Stochastic Neighbor Embedding):**
- Keeps similar points close together in lower dimensions
- Great for visualizing clusters
- NOT for feature extraction — only for visualization!
- Has perplexity hyperparameter (5–50 typical)

## 10. t-SNE Implementation

In [ ]:
from sklearn.manifold import TSNE
from sklearn.datasets import make_blobs

# Generate data with complex structure
X_tsne_data, y_tsne = make_blobs(n_samples=400, n_features=10,
                                   centers=4, cluster_std=2.0, random_state=42)
scaler_tsne = StandardScaler()
X_tsne_scaled = scaler_tsne.fit_transform(X_tsne_data)

print(f"Input shape: {X_tsne_scaled.shape}")

# PCA: 10D -> 2D
pca_comp = PCA(n_components=2, random_state=42)
X_pca_2d = pca_comp.fit_transform(X_tsne_scaled)
print(f"PCA variance explained: {pca_comp.explained_variance_ratio_.sum()*100:.1f}%")

# t-SNE: 10D -> 2D
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne_2d = tsne.fit_transform(X_tsne_scaled)

print(f"t-SNE done! KL divergence: {tsne.kl_divergence_:.4f}")
print("(Lower KL divergence = better fit)")

In [ ]:
# Comparison plot: PCA vs t-SNE
colors = ['royalblue', 'tomato', 'forestgreen', 'purple']
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, (data, title) in enumerate([(X_pca_2d, 'PCA (Linear)'), (X_tsne_2d, 't-SNE (Non-linear)')]):
    ax = axes[idx]
    for i, c in enumerate(np.unique(y_tsne)):
        mask = y_tsne == c
        ax.scatter(data[mask,0], data[mask,1], color=colors[i], alpha=0.7, s=30, label=f'Cluster {i}')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('PCA vs t-SNE: 10D → 2D', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print("t-SNE often gives better cluster separation for visualization!")
print("Note: t-SNE is NOT deterministic - results vary with random_state")

## 11. t-SNE Perplexity Effect

In [ ]:
# How perplexity affects t-SNE output
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
perplexities = [5, 30, 50]

for ax, perp in zip(axes, perplexities):
    tsne_p = TSNE(n_components=2, perplexity=perp, random_state=42, n_iter=1000)
    X_p = tsne_p.fit_transform(X_tsne_scaled)
    for i, c in enumerate(np.unique(y_tsne)):
        mask = y_tsne == c
        ax.scatter(X_p[mask,0], X_p[mask,1], color=colors[i], alpha=0.6, s=25)
    ax.set_title(f'perplexity={perp}', fontsize=12)
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.grid(True, alpha=0.3)

plt.suptitle('t-SNE Sensitivity to Perplexity', fontsize=13)
plt.tight_layout()
plt.show()
print("perplexity ~ number of effective neighbors to consider")
print("Low perplexity = local structure; High = more global")

## 12. Capstone Project — PCA Pipeline

**Task:** Apply complete PCA pipeline on synthetic high-dimensional customer data:
- 500 customers, 15 features
- Reduce to optimal dimensions
- Visualize clusters after PCA
- Compare reconstruction quality

In [ ]:
# ============================================================
# CAPSTONE: Complete PCA Pipeline
# ============================================================
from sklearn.cluster import KMeans
from sklearn.datasets import make_classification

np.random.seed(42)

# Synthetic customer data: 500 customers, 15 features
X_cap, y_cap = make_classification(n_samples=500, n_features=15,
                                     n_informative=6, n_redundant=4,
                                     n_classes=3, random_state=42)
print(f"Customer data shape: {X_cap.shape}")
print(f"Feature stats:\n  Mean: {X_cap.mean(axis=0)[:5].round(2)} ...")

# Step 1: Standardize
scaler_cap = StandardScaler()
X_cap_scaled = scaler_cap.fit_transform(X_cap)

# Step 2: Find optimal n_components (95% variance)
pca_cap_full = PCA().fit(X_cap_scaled)
cumvar = np.cumsum(pca_cap_full.explained_variance_ratio_)
n_opt = np.argmax(cumvar >= 0.95) + 1
print(f"\nOptimal components (95% var): {n_opt} (from 15 features)")

# Step 3: Apply optimal PCA
pca_cap = PCA(n_components=n_opt)
X_cap_pca = pca_cap.fit_transform(X_cap_scaled)
print(f"Compressed shape: {X_cap_pca.shape}")
print(f"Variance retained: {pca_cap.explained_variance_ratio_.sum()*100:.1f}%")

# Step 4: Reconstruct and check error
X_cap_recon = pca_cap.inverse_transform(X_cap_pca)
recon_error = np.mean((X_cap_scaled - X_cap_recon)**2)
print(f"Reconstruction MSE: {recon_error:.4f}")

In [ ]:
# Step 5: Visualize PCA 2D projection with true labels
pca_vis = PCA(n_components=2)
X_cap_2d = pca_vis.fit_transform(X_cap_scaled)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# PCA 2D colored by true labels
colors_cap = ['royalblue', 'tomato', 'forestgreen']
for i, c in enumerate(np.unique(y_cap)):
    mask = y_cap == c
    axes[0].scatter(X_cap_2d[mask,0], X_cap_2d[mask,1],
                    color=colors_cap[i], alpha=0.6, s=30, label=f'Class {i}')
axes[0].set_title('PCA: 15D → 2D (True Labels)', fontsize=11)
axes[0].set_xlabel(f'PC1 ({pca_vis.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca_vis.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# t-SNE 2D for comparison
tsne_cap = TSNE(n_components=2, perplexity=30, random_state=42)
X_cap_tsne = tsne_cap.fit_transform(X_cap_scaled)
for i, c in enumerate(np.unique(y_cap)):
    mask = y_cap == c
    axes[1].scatter(X_cap_tsne[mask,0], X_cap_tsne[mask,1],
                    color=colors_cap[i], alpha=0.6, s=30, label=f'Class {i}')
axes[1].set_title('t-SNE: 15D → 2D (True Labels)', fontsize=11)
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('PCA vs t-SNE on Customer Data', fontsize=13)
plt.tight_layout()
plt.show()

print("\n=== CAPSTONE SUMMARY ===")
print(f"Original dimensions: 15")
print(f"Compressed (95% var): {n_opt}")
print(f"Compression ratio: {15/n_opt:.1f}x")
print(f"Variance retained: {pca_cap.explained_variance_ratio_.sum()*100:.1f}%")
print(f"Reconstruction MSE: {recon_error:.4f}")
print("PCA done! Useful for feature reduction before ML models.")